In [6]:
# Cell 1 — Imports
import os
import json
import torch
import time
import torch.nn as nn
import mlflow
import mlflow.pytorch
import boto3
import queue
import threading
import random
from io import BytesIO
from PIL import Image
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, Dataset, random_split, Subset
from pathlib import Path
from dotenv import load_dotenv
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from pyinaturalist import get_taxa

print(f"PyTorch: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
# print(torch.cuda.get_device_name(0))

PyTorch: 2.12.0
MPS available: True


In [4]:
# Cell 2 — Config
load_dotenv(Path("../.env"))

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
BUCKET = 'fish-classifier-expanded'
PREFIX = 'raw/'
MODEL_SAVE_PATH = Path("../models/efficientnet_b4_fish_expanded.pth")
BATCH_SIZE = 32
EPOCHS = 15
LR = 0.001
IMG_SIZE = 224
MAX_WORKERS = 0

s3 = boto3.client(
    's3',
    endpoint_url=os.getenv('AISTOR_ENDPOINT'),
    aws_access_key_id=os.getenv('AISTOR_ACCESS_KEY'),
    aws_secret_access_key=os.getenv('AISTOR_SECRET_KEY')
)

mlflow.set_tracking_uri(os.getenv("MLFLOW_URI"))
mlflow.set_experiment("fish-classifier-expanded")

print(f"Device: {DEVICE}")
print(f"Bucket: {BUCKET}")

Device: mps
Bucket: fish-classifier-expanded


In [ ]:
# Cell 3 — AIStor Dataset with persistent thread pool
import atexit

_executor = ThreadPoolExecutor(max_workers=20)
atexit.register(_executor.shutdown)

class AiStorDataset(Dataset):
    def __init__(self, bucket, prefix, transform=None):
        self.bucket = bucket
        self.prefix = prefix
        self.transform = transform
        self.samples = []
        self.classes = []
        self.class_to_idx = {}

        print("Indexing AIStor dataset...")
        paginator = s3.get_paginator('list_objects_v2')
        class_set = set()
        pages = []

        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            pages.append(page)
            for obj in page.get('Contents', []):
                parts = obj['Key'].split('/')
                if len(parts) >= 3:
                    class_set.add(parts[1])

        self.classes = sorted(list(class_set))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

        for page in pages:
            for obj in page.get('Contents', []):
                key = obj['Key']
                parts = key.split('/')
                if len(parts) >= 3:
                    self.samples.append((key, self.class_to_idx[parts[1]]))

        print(f"Found {len(self.samples):,} images across {len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def _fetch(self, key):
        obj = s3.get_object(Bucket=self.bucket, Key=key)
        img = Image.open(BytesIO(obj['Body'].read())).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        key, label = self.samples[idx]
        img = self._fetch(key)
        return img, label

    def fetch_batch(self, indices):
        items = [self.samples[i] for i in indices]
        keys = [k for k, _ in items]
        labels = [l for _, l in items]
        imgs = list(_executor.map(self._fetch, keys))
        return torch.stack(imgs), torch.tensor(labels)

In [ ]:
# Cell 4 — Transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Transforms ready")

In [ ]:
# Cell 5 — Load dataset and split
full_dataset = AiStorDataset(
    bucket=BUCKET,
    prefix=PREFIX,
    transform=train_transforms
)
classes = full_dataset.classes
NUM_CLASSES = len(classes)

print(f"Total images: {len(full_dataset):,}")
print(f"Classes: {NUM_CLASSES}")

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

# Val and test use val_transforms
val_ds.dataset.transform = val_transforms
test_ds.dataset.transform = val_transforms

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=MAX_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=MAX_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=MAX_WORKERS)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

In [ ]:
# # Cell — Benchmark AIStor vs SMB
# test_transforms = transforms.Compose([
#     transforms.Resize((380, 380)),
#     transforms.ToTensor(),
# ])

# # ── Benchmark SMB ──
# print("Benchmarking SMB...")
# smb_dataset = datasets.ImageFolder(Path("/Volumes/kaggle/inat_fish"), transform=test_transforms)
# smb_subset = Subset(smb_dataset, list(range(320)))
# smb_loader = DataLoader(smb_subset, batch_size=32, shuffle=False, num_workers=4)

# start = time.time()
# for imgs, labels in smb_loader:
#     pass
# smb_time = time.time() - start
# print(f"SMB:    {smb_time:.2f}s | {320/smb_time:.1f} img/s")

# # ── Benchmark AIStor ──
# print("\nBenchmarking AIStor...")

# # Get 320 keys
# paginator = s3.get_paginator('list_objects_v2')
# keys = []
# for page in paginator.paginate(Bucket='fish-classifier-expanded', Prefix='raw/', PaginationConfig={'MaxItems': 320}):
#     for obj in page.get('Contents', []):
#         keys.append(obj['Key'])
#     if len(keys) >= 320:
#         break

# def fetch_image(key):
#     obj = s3.get_object(Bucket='fish-classifier-expanded', Key=key)
#     img = Image.open(BytesIO(obj['Body'].read())).convert('RGB')
#     return test_transforms(img)

# start = time.time()
# with ThreadPoolExecutor(max_workers=20) as executor:
#     results = list(executor.map(fetch_image, keys[:320]))
# aistor_time = time.time() - start
# print(f"AIStor: {aistor_time:.2f}s | {320/aistor_time:.1f} img/s")

# print(f"\nWinner: {'SMB' if smb_time < aistor_time else 'AIStor'} is {max(smb_time, aistor_time)/min(smb_time, aistor_time):.1f}x faster")

In [ ]:
# Cell 6 — Model
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f"Model ready — output classes: {NUM_CLASSES}")

In [ ]:
# Cell 7 — Training loop with prefetching
def prefetch_batches(dataset, indices, batch_size, q, n_prefetch=2):
    for i in range(0, len(indices), batch_size):
        batch_indices = indices[i:i+batch_size]
        imgs, labels = dataset.fetch_batch(batch_indices)
        q.put((imgs, labels))
    q.put(None)

def run_epoch(dataset, indices, batch_size):
    q = queue.Queue(maxsize=3)
    t = threading.Thread(target=prefetch_batches, args=(dataset, indices, batch_size, q))
    t.start()
    while True:
        batch = q.get()
        if batch is None:
            break
        yield batch
    t.join()

with mlflow.start_run(run_name="efficientnet_b4_202species"):
    mlflow.log_params({
        "model": "efficientnet_b4",
        "num_classes": NUM_CLASSES,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "img_size": IMG_SIZE,
        "device": str(DEVICE),
        "data_source": "AIStor"
    })

    train_indices = list(train_ds.indices)
    val_indices = list(val_ds.indices)

    for epoch in range(EPOCHS):
        # ── Train ──
        model.train()
        train_loss, train_correct = 0, 0
        random.shuffle(train_indices)
        n_batches = len(train_indices) // BATCH_SIZE

        with tqdm(total=n_batches, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", unit="batch") as pbar:
            for imgs, labels in run_epoch(full_dataset, train_indices, BATCH_SIZE):
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
                train_correct += (outputs.argmax(1) == labels).sum().item()
                pbar.set_postfix({
                    'loss': f"{loss.item():.4f}",
                    'acc': f"{train_correct / ((pbar.n + 1) * BATCH_SIZE):.4f}"
                })
                pbar.update(1)

        train_acc = train_correct / len(train_indices)

        # ── Validate ──
        model.eval()
        val_correct = 0
        n_val_batches = len(val_indices) // BATCH_SIZE

        with torch.no_grad():
            with tqdm(total=n_val_batches, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  ", unit="batch") as pbar:
                for imgs, labels in run_epoch(full_dataset, val_indices, BATCH_SIZE):
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    outputs = model(imgs)
                    val_correct += (outputs.argmax(1) == labels).sum().item()
                    pbar.set_postfix({'acc': f"{val_correct / ((pbar.n + 1) * BATCH_SIZE):.4f}"})
                    pbar.update(1)

        val_acc = val_correct / len(val_indices)
        scheduler.step()

        mlflow.log_metrics({
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "train_loss": train_loss / len(train_indices) * BATCH_SIZE
        }, step=epoch)

        print(f"Epoch {epoch+1}/{EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}", flush=True)

    # ── Save model — inside with block ──
    torch.save({
        'model_state_dict': model.state_dict(),
        'classes': classes,
        'num_classes': NUM_CLASSES
    }, MODEL_SAVE_PATH)

    mlflow.log_artifact(str(MODEL_SAVE_PATH))
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
        registered_model_name="fishid-efficientnet-b4"
    )

    print(f"\nModel saved to {MODEL_SAVE_PATH}", flush=True)

In [5]:
# Load classes from saved model
checkpoint = torch.load(MODEL_SAVE_PATH, map_location='cpu')
classes = checkpoint['classes']
print(f"Loaded {len(classes)} classes from checkpoint")

# Query iNaturalist for common names
results = get_taxa(
    taxon_id=47178,
    place_id=59,
    rank="species",
    per_page=200,
    order_by="observations_count",
    order="desc"
)

# Build scientific → common name lookup
name_lookup = {}
for taxon in results['results']:
    scientific = taxon['name'].replace(' ', '_')
    common = taxon.get('preferred_common_name', '')
    if common:
        name_lookup[scientific] = common

# Add mackerel manually
name_lookup['Scomber_japonicus'] = 'Pacific Chub Mackerel'
name_lookup['Trachurus_symmetricus'] = 'Pacific Jack Mackerel'

# Map all classes
class_to_common = {}
missing = []
for cls in classes:
    if cls in name_lookup:
        class_to_common[cls] = name_lookup[cls]
    else:
        class_to_common[cls] = cls.replace('_', ' ')
        missing.append(cls)

# Save
output_path = MODEL_SAVE_PATH.parent / 'class_to_common.json'
with open(output_path, 'w') as f:
    json.dump(class_to_common, f, indent=2)

print(f"Saved {len(class_to_common)} mappings to {output_path}")
print(f"Missing common names: {len(missing)}")
for m in missing[:10]:
    print(f"  {m}")

Loaded 202 classes from checkpoint
Saved 202 mappings to ../models/class_to_common.json
Missing common names: 0
